# Orbit Wars Agent Benchmark Notebook

Notebook này dùng để nạp nhiều agent từ các file `.py` riêng, cho 4 agent đấm nhau trong môi trường ngẫu nhiên, rồi chốt agent thắng để copy sang `main.py` trước khi submit.

Ghi chú:
- Không set seed để môi trường vẫn ngẫu nhiên.
- Phần 2-agent battle và phần submit được để comment theo đúng yêu cầu.
- Các file agent được đặt cùng thư mục với notebook, không dùng folder `agents/`.
- Đồng bộ GitHub là tùy chọn, không bắt buộc để notebook chạy.
- Mỗi agent chỉ cần expose hàm `agent(obs)`.

In [ ]:
import shutil
from pathlib import Path

from importlib.machinery import SourceFileLoader
from importlib.util import module_from_spec, spec_from_loader

from kaggle_environments import make

ROOT = Path.cwd()

def load_agent_from_file(file_path):
    file_path = Path(file_path).resolve()
    module_name = f"orbit_wars_{file_path.stem}_{abs(hash(str(file_path))) & 0xffffffff}"
    loader = SourceFileLoader(module_name, str(file_path))
    spec = spec_from_loader(module_name, loader)
    module = module_from_spec(spec)
    loader.exec_module(module)
    if not hasattr(module, "agent"):
        raise AttributeError(f"{file_path} must define an agent(obs) function")
    return module.agent

def promote_agent_to_main(source_file, target_file="main.py"):
    source_path = Path(source_file).resolve()
    target_path = Path(target_file).resolve()
    shutil.copyfile(source_path, target_path)
    print(f"Copied {source_path} -> {target_path}")

def play_match(agent_files, debug=True):
    agent_funcs = [load_agent_from_file(path) for path in agent_files]
    env = make("orbit_wars", debug=debug)
    env.run(agent_funcs)
    final_states = env.steps[-1]
    rewards = [state.reward for state in final_states]
    statuses = [state.status for state in final_states]
    return env, rewards, statuses

In [ ]:
# GitHub sync (Kaggle-ready).
# Set REPO_URL and BRANCH. For private repos, add GITHUB_TOKEN in Kaggle Secrets.

import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/huyvanzzz/Orbit-Wars.git"
BRANCH = "main"
REPO_DIR = Path("repo")

def _run(cmd, cwd=None):
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr)
        raise RuntimeError("Git command failed: " + " ".join(cmd))
    return result

def _with_token(url):
    token = os.environ.get("GITHUB_TOKEN")
    if not token:
        return url
    if url.startswith("https://"):
        return url.replace("https://", f"https://{token}@")
    return url

def sync_repo():
    if (REPO_DIR / ".git").exists():
        _run(["git", "fetch", "origin"], cwd=str(REPO_DIR))
        _run(["git", "checkout", BRANCH], cwd=str(REPO_DIR))
        _run(["git", "pull", "origin", BRANCH], cwd=str(REPO_DIR))
    else:
        url = _with_token(REPO_URL)
        _run(["git", "clone", "--branch", BRANCH, url, str(REPO_DIR)])
    return REPO_DIR

AGENT_ROOT = sync_repo()
print("Using agent root:", AGENT_ROOT)

In [ ]:
# Ensure Kaggle environment supports orbit_wars.
# Run this once per session if you see: InvalidArgument: Unknown Environment Specification

import sys
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "-U", "-q", "kaggle-environments>=1.28.0"], check=True)

from kaggle_environments import make
import kaggle_environments as ke

print("kaggle_environments version:", getattr(ke, "__version__", "unknown"))
try:
    _ = make("orbit_wars", debug=False)
    print("orbit_wars available")
except Exception as exc:
    print("orbit_wars not available; restart session after upgrade if needed")
    raise exc

In [ ]:
# 4-agent battle is enabled here.
# Edit these paths to point at the four agent files you want to compare.
AGENT_FILES = [
    str(AGENT_ROOT / "agent_01.py"),
    str(AGENT_ROOT / "agent_02.py"),
    str(AGENT_ROOT / "agent_03.py"),
    str(AGENT_ROOT / "agent_04.py"),
]

env, rewards, statuses = play_match(AGENT_FILES, debug=True)
print("Rewards:", rewards)
print("Statuses:", statuses)
env.render(mode="ipython", width=800, height=600)

In [ ]:
# 4-agent round-robin / nhiều ván ngẫu nhiên để so điểm trung bình.
# Không set seed để mỗi ván có độ ngẫu nhiên riêng.
from collections import defaultdict

N_MATCHES = 10
scoreboard = defaultdict(list)

for _ in range(N_MATCHES):
    env, rewards, statuses = play_match(AGENT_FILES, debug=False)
    for path, reward in zip(AGENT_FILES, rewards):
        scoreboard[Path(path).stem].append(reward)

for name, rewards_list in sorted(scoreboard.items()):
    avg_reward = sum(rewards_list) / len(rewards_list)
    print(name, "avg_reward=", round(avg_reward, 4), "matches=", len(rewards_list))

In [ ]:
# 2-agent battle - comment lại theo yêu cầu.
# Bật cell này nếu muốn test chỉ 2 agent.
#
# agent_a = load_agent_from_file("agent_01.py")
# agent_b = load_agent_from_file("agent_02.py")
#
# N_MATCHES = 10
# score_a = []
# score_b = []
#
# for match_index in range(N_MATCHES):
#     env = make("orbit_wars", debug=True)
#     if match_index % 2 == 0:
#         env.run([agent_a, agent_b])
#     else:
#         env.run([agent_b, agent_a])
#
#     final_rewards = [state.reward for state in env.steps[-1]]
#     if match_index % 2 == 0:
#         score_a.append(final_rewards[0])
#         score_b.append(final_rewards[1])
#     else:
#         score_a.append(final_rewards[1])
#         score_b.append(final_rewards[0])
#
# print("agent_a avg_reward=", sum(score_a) / len(score_a))
# print("agent_b avg_reward=", sum(score_b) / len(score_b))
# env.render(mode="ipython", width=800, height=600)

In [ ]:
# 1 lệnh để biến 1 file agent thành file main.py cố định.
# Chỉ cần đổi source_file sang agent bạn muốn chốt.
#
# promote_agent_to_main("agent_04.py")

In [ ]:
# Submit - comment lại theo yêu cầu.
# Khi đã chốt agent thắng, copy nó về main.py rồi mới submit.
#
# !kaggle competitions submit orbit-wars -f main.py -m "Orbit Wars agent v1"